# Project Lab 1, Prediction, Classification, Fraud and Competition Workflow

Notebook for **Chapter 17** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch17.html

Tip: change a number, predict the output, then run the cell again.

## 17.1 How to run a project in this area

## 17.2 Project: Prediction With a Tuned Model and an Explanation (House-Price Style)

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score

def make_houses(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    area = rng.normal(120, 35, n).clip(35, 300)
    rooms = np.clip((area/35 + rng.normal(0, .6, n)).round(), 1, 8)
    age = rng.uniform(0, 60, n)
    dist = rng.gamma(3, 3, n)
    zone = rng.choice(["north", "centre", "south"], n, p=[.35, .25, .4])
    zone_eff = pd.Series(zone).map({"north": 0, "centre": 30, "south": -10}).to_numpy()
    price = (20 + 1.1*area + 4*rooms - 0.5*age - 2.0*dist + zone_eff
             + 0.004*area*np.maximum(0, 15-dist)*3 + rng.normal(0, 12, n)).clip(15, None)
    return pd.DataFrame(dict(area=area, rooms=rooms, age=age, dist=dist, zone=zone, price=price))

df = make_houses()
df["zone"] = df["zone"].astype("category")
X, y = df.drop(columns="price"), df["price"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(df.describe().round(1).loc[["mean","std","min","max"]])
print("missing:", int(df.isna().sum().sum()))

In [ ]:
def report(name, m):
    p = m.predict(X_te)
    print(f"{name:22s} MAE={mean_absolute_error(y_te,p):6.2f}  R2={r2_score(y_te,p):.3f}")

base = DummyRegressor(strategy="median").fit(X_tr, y_tr); report("median baseline", base)
Xd = pd.get_dummies(X_tr, columns=["zone"]).astype(float); Xtd = pd.get_dummies(X_te, columns=["zone"]).astype(float)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(Xd, y_tr)
p = ridge.predict(Xtd)
print(f"{'ridge':22s} MAE={mean_absolute_error(y_te,p):6.2f}  R2={r2_score(y_te,p):.3f}")
hgb = HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0).fit(X_tr, y_tr); report("hgb default", hgb)
space = dict(learning_rate=[0.03,0.05,0.1,0.2], max_leaf_nodes=[8,15,31], min_samples_leaf=[10,20,40,80],
             l2_regularization=[0,1,10], max_iter=[150,300,600])
rs = RandomizedSearchCV(HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0),
                        space, n_iter=15, cv=5, scoring="neg_mean_absolute_error", random_state=0, n_jobs=1).fit(X_tr, y_tr)
print("best params:", rs.best_params_, " cv MAE=%.2f" % -rs.best_score_)
report("hgb tuned (test)", rs.best_estimator_)

median baseline        MAE= 42.81  R2=-0.002
ridge                  MAE= 10.00  R2=0.945
hgb default            MAE= 10.87  R2=0.934
best params: {'min_samples_leaf': 20, 'max_leaf_nodes': 8, 'max_iter': 300, 'learning_rate': 0.03, 'l2_regularization': 1}  cv MAE=10.17
hgb tuned (test)       MAE= 10.62  R2=0.938


In [ ]:
pi = permutation_importance(rs.best_estimator_, X_te, y_te, n_repeats=10, random_state=0, scoring="neg_mean_absolute_error")
for i in pi.importances_mean.argsort()[::-1]: print(f"  {X.columns[i]:6s} {pi.importances_mean[i]:6.2f} +- {pi.importances_std[i]:.2f}")

  area    36.83 +- 0.62
  dist     9.90 +- 0.52
  zone     9.65 +- 0.61
  age      4.10 +- 0.36
  rooms    1.51 +- 0.14


In [ ]:
res = y_te - rs.best_estimator_.predict(X_te)
print("residual mean %.2f, std %.2f" % (res.mean(), res.std()))
print(res.groupby(X_te["zone"], observed=True).agg(["mean","std"]).round(2))

residual mean 0.06, std 13.14
        mean    std
zone               
centre  3.72  12.35
north  -0.85  12.76
south  -1.42  13.66


## 17.3 Project: Classification With Calibration and a Cost-Based Threshold (Churn Style)

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def make_customers(n=8000, seed=1):
    rng = np.random.default_rng(seed)
    tenure = rng.integers(1, 61, n); monthly = rng.normal(60, 20, n).clip(15, 150)
    tickets = rng.poisson(1.2, n); late = rng.binomial(1, .25, n)
    monthly_contract = rng.binomial(1, .6, n)
    z = -0.6 - 0.04*tenure + 0.015*(monthly-60) + 0.45*tickets + 0.7*late + 0.9*monthly_contract
    churn = rng.binomial(1, 1/(1+np.exp(-z)))
    return pd.DataFrame(dict(tenure=tenure, monthly=monthly, tickets=tickets, late=late,
                             monthly_contract=monthly_contract, churn=churn))
df = make_customers(); X, y = df.drop(columns="churn"), df.churn
print("churn rate", round(y.mean(), 3))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, stratify=y, random_state=0)

def report(name, p):
    print(f"{name:22s} AUC={roc_auc_score(y_te,p):.3f} Brier={brier_score_loss(y_te,p):.4f} LogLoss={log_loss(y_te,p):.4f}")
make_lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
make_gb = lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=20, random_state=0)
lr = make_lr().fit(X_tr, y_tr);  p_lr = lr.predict_proba(X_te)[:, 1];  report("logistic", p_lr)
gb = make_gb().fit(X_tr, y_tr);  p_gb = gb.predict_proba(X_te)[:, 1];  report("boosting (raw)", p_gb)
iso = CalibratedClassifierCV(make_gb(), method="isotonic", cv=5).fit(X_tr, y_tr)
p_iso = iso.predict_proba(X_te)[:, 1];                                   report("boosting + isotonic", p_iso)

churn rate 0.383
logistic               AUC=0.750 Brier=0.1930 LogLoss=0.5671
boosting (raw)         AUC=0.702 Brier=0.2171 LogLoss=0.6420
boosting + isotonic    AUC=0.702 Brier=0.2076 LogLoss=0.6015


In [ ]:
for name, p in [("raw boosting", p_gb), ("isotonic", p_iso)]:
    obs, mean_pred = calibration_curve(y_te, p, n_bins=5, strategy="quantile")
    print(name.ljust(13), " ".join(f"({m:.2f}->{o:.2f})" for m, o in zip(mean_pred, obs)))

raw boosting  (0.06->0.17) (0.20->0.29) (0.35->0.33) (0.53->0.49) (0.79->0.62)
isotonic      (0.17->0.17) (0.29->0.29) (0.38->0.36) (0.47->0.47) (0.62->0.63)


In [ ]:
C_FN, C_FP = 60, 15            # missing a churner costs 60; a wasted offer costs 15
def cost(y_true, p, t):
    pred = p >= t; fn = ((~pred) & (y_true == 1)).sum(); fp = (pred & (y_true == 0)).sum()
    return C_FN*fn + C_FP*fp
grid = np.linspace(.02, .98, 97)
oof = cross_val_predict(make_lr(), X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
t_best = grid[np.argmin([cost(y_tr, oof, t) for t in grid])]
print("theory t* =", round(C_FP/(C_FP+C_FN), 3), "| chosen on OOF t =", round(t_best, 2))
print("TEST cost: t=0.5 ->", cost(y_te, p_lr, .5), "| t=chosen ->", cost(y_te, p_lr, t_best),
      "| flag nobody ->", C_FN*int(y_te.sum()), "| flag everybody ->", C_FP*int((1-y_te).sum()))

theory t* = 0.2 | chosen on OOF t = 0.21
TEST cost: t=0.5 -> 25395 | t=chosen -> 15930 | flag nobody -> 45900 | flag everybody -> 18525


## 17.4 Project: Fraud and Anomaly Detection With Extreme Imbalance and Time-Aware Validation

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

def make_tx(n=120_000, days=90, seed=7):
    rng = np.random.default_rng(seed)
    day = np.sort(rng.integers(0, days, n))
    hour = rng.integers(0, 24, n)
    amount = rng.lognormal(3.3, 1.0, n)
    card = rng.integers(0, 6000, n)
    foreign = rng.binomial(1, .05, n)
    night = (hour < 5).astype(int)
    z = -7.0 + 1.6*night + 2.0*foreign
    late = day >= 60
    z = z + late*(-1.5*night - 1.5*foreign + 1.4*np.log(amount/27)*(amount > 150) + 2.0*(amount > 150))
    fraud = rng.binomial(1, 1/(1+np.exp(-z)))
    df = pd.DataFrame(dict(day=day, hour=hour, amount=amount, card=card, foreign=foreign, night=night, fraud=fraud))
    df["card_prior_tx"] = df.groupby("card").cumcount()     # past-only: earlier transactions of this card
    df["log_amt"] = np.log1p(df.amount)
    return df
df = make_tx()
feats = ["hour","log_amt","foreign","night","card_prior_tx"]
print("rows", len(df), " fraud rate %.4f" % df.fraud.mean())
print(df.groupby(pd.cut(df.day,[-1,29,59,89])).fraud.agg(["size","sum","mean"]).round(4))

rows 120000  fraud rate 0.0040
           size  sum    mean
day                         
(-1, 29]  39950   92  0.0023
(29, 59]  39835  107  0.0027
(59, 89]  40215  284  0.0071


In [ ]:
def fit_eval(tr, te, name):
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=.05, max_leaf_nodes=15,
                                       min_samples_leaf=40, random_state=0).fit(tr[feats], tr.fraud)
    s = m.predict_proba(te[feats])[:,1]
    k = int(0.01*len(te)); top = np.argsort(-s)[:k]
    print(f"{name:34s} PR-AUC={average_precision_score(te.fraud,s):.3f} ROC-AUC={roc_auc_score(te.fraud,s):.3f} "
          f"recall@1%={te.fraud.values[top].sum()/te.fraud.sum():.3f} (frauds in test={te.fraud.sum()})")

In [ ]:
train = df[df.day < 60]; test = df[df.day >= 60]
rtr, rte = train_test_split(df, test_size=.3, stratify=df.fraud, random_state=0)
fit_eval(rtr, rte, "random split (looks great, is not)")
fit_eval(train, test, "time split: train<60, test>=60")
tr2, va2 = train[train.day < 45], train[train.day >= 45]
fit_eval(tr2, va2, "time validation: <45 -> 45..59")
te2 = df[df.day >= 75]
fit_eval(df[df.day < 75], te2, "all <75 -> test 75..89")
fit_eval(df[(df.day >= 30) & (df.day < 75)], te2, "last 45 days (30..74) -> 75..89")
fit_eval(df[df.day < 60], te2, "stale (<60) -> test 75..89")

random split (looks great, is not) PR-AUC=0.133 ROC-AUC=0.859 recall@1%=0.359 (frauds in test=145)
time split: train<60, test>=60     PR-AUC=0.011 ROC-AUC=0.624 recall@1%=0.018 (frauds in test=284)
time validation: <45 -> 45..59     PR-AUC=0.012 ROC-AUC=0.701 recall@1%=0.109 (frauds in test=55)
all <75 -> test 75..89             PR-AUC=0.150 ROC-AUC=0.934 recall@1%=0.274 (frauds in test=135)
last 45 days (30..74) -> 75..89    PR-AUC=0.157 ROC-AUC=0.920 recall@1%=0.281 (frauds in test=135)
stale (<60) -> test 75..89         PR-AUC=0.011 ROC-AUC=0.641 recall@1%=0.007 (frauds in test=135)


## 17.5 Project: A Kaggle-Style Competition Workflow (Experiment Log, Cross-Validation, Ensembling)

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def make_orders(n=6000, seed=3):
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(dict(distance=rng.gamma(2, 6, n), weight=rng.lognormal(1, .6, n), n_items=rng.integers(1, 12, n),
                          hour=rng.integers(8, 22, n), rain=rng.binomial(1, .2, n), courier_exp=rng.uniform(0, 5, n),
                          noise1=rng.normal(size=n), noise2=rng.normal(size=n)))
    z = (-2.0 + 0.09*X.distance + 0.5*X.rain - 0.25*X.courier_exp + 0.25*(X.hour.between(17, 20))*X.n_items/3
         + 0.6*np.sin(X.weight*1.5) + 0.04*X.distance*X.rain*2)
    return X, rng.binomial(1, 1/(1+np.exp(-z)))
X, y = make_orders()
print("late rate %.3f" % y.mean())
Xd, Xh, yd, yh = train_test_split(X, y, test_size=.2, stratify=y, random_state=0)
Xd, Xh = Xd.reset_index(drop=True), Xh.reset_index(drop=True)

models = {
 "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
 "rf": RandomForestClassifier(300, min_samples_leaf=10, n_jobs=1, random_state=0),
 "extra": ExtraTreesClassifier(300, min_samples_leaf=10, n_jobs=1, random_state=0),
 "hgb": HistGradientBoostingClassifier(learning_rate=.03, max_iter=250, max_leaf_nodes=8, min_samples_leaf=30, random_state=0),
}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof = {}; log = []
for name, m in models.items():
    o = np.zeros(len(yd)); fold_auc = []
    for tr, va in skf.split(Xd, yd):
        m.fit(Xd.iloc[tr], yd[tr]); o[va] = m.predict_proba(Xd.iloc[va])[:,1]; fold_auc.append(roc_auc_score(yd[va], o[va]))
    oof[name] = o
    log.append(dict(exp=name, oof_auc=roc_auc_score(yd, o), fold_std=np.std(fold_auc)))

In [ ]:
def fe(df):
    d = df.copy(); d["dist_rain"] = d.distance*d.rain; d["rush"] = d.hour.between(17,20).astype(int)
    d["rush_items"] = d.rush*d["n_items"]; d["sinw"] = np.sin(1.5*d.weight); return d
o = np.zeros(len(yd)); fa = []
for tr, va in skf.split(Xd, yd):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(fe(Xd).iloc[tr], yd[tr])
    o[va] = m.predict_proba(fe(Xd).iloc[va])[:,1]; fa.append(roc_auc_score(yd[va], o[va]))
oof["logreg_fe"] = o; log.append(dict(exp="logreg_fe", oof_auc=roc_auc_score(yd, o), fold_std=np.std(fa)))
names = ["logreg_fe","rf","hgb"]
avg = np.mean([oof[k] for k in names], axis=0)
log.append(dict(exp="blend_avg(logreg_fe,rf,hgb)", oof_auc=roc_auc_score(yd, avg), fold_std=np.nan))
rk = np.mean([rankdata(oof[k]) for k in names], axis=0)
log.append(dict(exp="blend_rank", oof_auc=roc_auc_score(yd, rk), fold_std=np.nan))
print(pd.DataFrame(log).round(4).to_string(index=False))

late rate 0.285
                        exp  oof_auc  fold_std
                     logreg   0.7444    0.0189
                         rf   0.7532    0.0100
                      extra   0.7445    0.0183
                        hgb   0.7622    0.0128
                  logreg_fe   0.7737    0.0142
blend_avg(logreg_fe,rf,hgb)   0.7687       NaN
                 blend_rank   0.7682       NaN


In [ ]:
fm = {"logreg_fe": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), "rf": models["rf"], "hgb": models["hgb"]}
ph = {}
for k, m in fm.items():
    A, B = (fe(Xd), fe(Xh)) if k=="logreg_fe" else (Xd, Xh)
    m.fit(A, yd); ph[k] = m.predict_proba(B)[:,1]; print(f"holdout {k:10s} AUC={roc_auc_score(yh, ph[k]):.4f}")
print("holdout blend_avg AUC=%.4f" % roc_auc_score(yh, np.mean(list(ph.values()),axis=0)))

holdout logreg_fe  AUC=0.7776
holdout rf         AUC=0.7771
holdout hgb        AUC=0.7776
holdout blend_avg AUC=0.7821


## 17.6 Scaffolding a Project: Brief, Baseline Ladder and Experiment Log

The projects in 17.2 to 17.5 are finished examples. The hard part when you start your own is the blank page. This section is the scaffold: a one-page brief you fill in before any code, a baseline ladder that tells you when to stop, and an experiment log that makes your README honest.

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import average_precision_score
X, y = make_classification(3000, 10, n_informative=5, weights=[0.9], random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0); log = []
for name, m in [("prior (dummy)", DummyClassifier(strategy="prior")), ("logistic", LogisticRegression(max_iter=1000)), ("hist-GB", HistGradientBoostingClassifier(random_state=0))]:
    p = cross_val_predict(m, X, y, cv=cv, method="predict_proba")[:, 1]
    log.append({"id": len(log) + 1, "model": name, "PR-AUC": round(average_precision_score(y, p), 3)})
df = pd.DataFrame(log); df["lift_vs_baseline"] = (df["PR-AUC"] / df["PR-AUC"][0]).round(2)
print(df.to_string(index=False))

 id         model  PR-AUC  lift_vs_baseline
  1 prior (dummy)   0.103              1.00
  2      logistic   0.816              7.92
  3       hist-GB   0.875              8.50


## Chapter recap

## Practice set

**1. (easy)** Missing a churner costs 100 and a wasted offer costs 10. What is the best threshold in theory? **Answer:** `t* = 10 / (10 + 100) = 0.0909`. Flag customers with at least about 9 percent churn probability.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch17.html